# 马尔可夫链旋律生成

本 Notebook 包含：

1. **一阶马尔可夫链**：从本地按艺术家整理、来源映射未核实的 MIDI 子集统计音高转移概率，绘制 5×5 热力图
2. **高阶马尔可夫链**：对比 k=1, 2, 3 阶的生成效果
3. **评估**：在《茉莉花》上训练（绝对音高，时值）联合 token 模型，并比较生成结果

数据来源：
- 训练：`CODE/datasets/lmd_clean_midi/`（本地按西方艺术家命名的目录；来源映射与许可证未核实）
- 单曲实验：`CODE/datasets/melodies/茉莉花.midi`（具体谱源与版本未核实）

图片输出：`CODE/chapter04/output_figures/`（600 dpi）
MIDI 输出：`CODE/chapter04/output_midi/`

## 0. 环境与配置

In [ ]:
import os
import random
from collections import defaultdict, Counter

import numpy as np
import matplotlib.pyplot as plt
import pretty_midi

# 中文字体
plt.rcParams['font.sans-serif'] = [
    'PingFang SC', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei',
    'Arial Unicode MS', 'Noto Sans CJK SC', 'DejaVu Sans',
]
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['text.color'] = 'black'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

_p = os.getcwd()
while not os.path.exists(os.path.join(_p, 'CODE', 'datasets')):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p

DATASET_DIR = os.path.join(BASE_DIR, 'CODE', 'datasets')
LMD_DIR = os.path.join(DATASET_DIR, 'lmd_clean_midi')
MELODY_DIR = os.path.join(DATASET_DIR, 'melodies')
FIGURES_DIR = os.path.join(BASE_DIR, 'CODE', 'chapter04', 'output_figures')
MIDI_OUT_DIR = os.path.join(BASE_DIR, 'CODE', 'chapter04', 'output_midi')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MIDI_OUT_DIR, exist_ok=True)

# 五声音阶音级（pitch class）——本例使用的两个约束集合
PENTATONIC_PC = [0, 2, 4, 7, 9]      # C D E G A（本章按 C 宫五声音阶记名）
JASMINE_PENTA_PC = [0, 2, 5, 7, 9]   # 本地《茉莉花》MIDI 实测音级集合 C D F G A
PENTATONIC_NAMES = ['宫(C)', '商(D)', '角(E)', '徵(G)', '羽(A)']
JASMINE_PENTA_NAMES = ['C', 'D', 'F', 'G', 'A']

PC_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def pc_seq_to_names(pcs):
    return [PC_NAMES[pc] for pc in pcs]

print('BASE_DIR:', BASE_DIR)
print('MIDI 输出目录:', MIDI_OUT_DIR)

## 1. 工具函数：MIDI 解析与音高序列提取

In [ ]:
def _is_integer_scalar(value):
    return isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_))


def _expect_value_error(function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except ValueError:
        return
    raise AssertionError(f'{function.__name__} 对非法输入未抛出 ValueError')


def load_midi_notes(fpath, skip_drums=True):
    """按 MIDI tick 聚合同 onset 音符，并取最高音作为顶线启发式。"""
    midi_obj = pretty_midi.PrettyMIDI(fpath)
    onset_groups = defaultdict(list)
    for inst in midi_obj.instruments:
        if skip_drums and inst.is_drum:
            continue
        for n in inst.notes:
            onset_tick = int(round(midi_obj.time_to_tick(n.start)))
            onset_groups[onset_tick].append(n.pitch)
    return [(max(onset_groups[tick]), tick) for tick in sorted(onset_groups)]


def extract_pitch_sequence(fpath, skip_drums=True):
    """提取顶线启发式音高序列（仅 MIDI pitch 值），用于马尔可夫链训练。"""
    note_list = load_midi_notes(fpath, skip_drums)
    return [p for p, _ in note_list]


def extract_pitch_class_sequence(fpath, skip_drums=True):
    """提取音级序列（0-11），用于转移矩阵统计。"""
    pitch_list = extract_pitch_sequence(fpath, skip_drums)
    return [p % 12 for p in pitch_list]


def load_pitch_sequences_from_dir(directory, max_files=None, min_notes=10):
    """从目录批量加载 MIDI，返回音级序列列表。"""
    if not os.path.exists(directory):
        return []
    result = []
    midi_files = [
        fname for fname in sorted(os.listdir(directory))
        if fname.lower().endswith(('.mid', '.midi'))
    ]
    if max_files is not None:
        midi_files = midi_files[:max_files]
    for fname in midi_files:
        fpath = os.path.join(directory, fname)
        try:
            pc_seq = extract_pitch_class_sequence(fpath)
            if len(pc_seq) >= min_notes:
                result.append(pc_seq)
        except (IOError, ValueError, KeyError, EOFError):
            continue
    return result


def pitch_classes_to_midi(pitch_classes, base_octave=5, note_duration=0.3,
                          velocity=80, out_path=None):
    """将音级序列转换为 MIDI 文件并保存。

    参数：
        pitch_classes: 音级列表（0-11）
        base_octave: 科学音高记法中的八度编号（默认 C5=72）
        note_duration: 每个音的时长（秒）
        velocity: 力度
        out_path: 保存路径（None 则不保存）
    返回：
        pretty_midi.PrettyMIDI 对象
    """
    pitch_classes = list(pitch_classes)
    if any(not _is_integer_scalar(pc) or not 0 <= int(pc) < 12 for pc in pitch_classes):
        raise ValueError('pitch_classes 必须只包含 0--11 的整数音级。')
    if not _is_integer_scalar(base_octave):
        raise ValueError('base_octave 必须是整数。')
    if isinstance(note_duration, (bool, np.bool_)):
        raise ValueError('note_duration 必须是有限正数。')
    try:
        note_duration = float(note_duration)
    except (TypeError, ValueError) as exc:
        raise ValueError('note_duration 必须是有限正数。') from exc
    if not np.isfinite(note_duration) or note_duration <= 0:
        raise ValueError('note_duration 必须是有限正数。')
    if not _is_integer_scalar(velocity) or not 0 <= int(velocity) <= 127:
        raise ValueError('velocity 必须是 0--127 的整数。')

    midi_pitches = [(int(base_octave) + 1) * 12 + int(pc) for pc in pitch_classes]
    if any(not 0 <= midi_pitch <= 127 for midi_pitch in midi_pitches):
        raise ValueError('base_octave 与音级组合必须落在 MIDI pitch 0--127。')

    midi_obj = pretty_midi.PrettyMIDI(initial_tempo=120)
    inst = pretty_midi.Instrument(program=0)  # 钢琴
    onset = 0.0
    for midi_pitch in midi_pitches:
        n = pretty_midi.Note(
            velocity=velocity, pitch=midi_pitch,
            start=onset, end=onset + note_duration
        )
        inst.notes.append(n)
        onset += note_duration
    midi_obj.instruments.append(inst)
    if out_path is not None:
        midi_obj.write(out_path)
        print(f'  MIDI 已保存: {out_path}')
    return midi_obj


# 科学音高记法与 MIDI 的换算：C5 = 12 * (5 + 1) = 72。
_c5_check = pitch_classes_to_midi([0], base_octave=5)
assert _c5_check.instruments[0].notes[0].pitch == 72
_expect_value_error(pitch_classes_to_midi, [12])
_expect_value_error(pitch_classes_to_midi, [0], note_duration=0)
_expect_value_error(pitch_classes_to_midi, [0], velocity=128)

print('工具函数与边界检查定义完毕。')

## 2. 一阶马尔可夫链

### 2.1 从本地艺术家目录子集统计转移矩阵

从多位作曲家的 MIDI 文件中提取顶线启发式音高序列：排除鼓轨，按 MIDI tick 合并同 onset 事件，并取该 onset 的最高音。这样避免把同一和弦内任意文件顺序误当作时间转移；但它仍不是经过声部分离或人工旋律标注的真值。随后统计相邻 onset 之间的音级转移频次，并按行归一化为转移概率矩阵。

In [ ]:
# 从多位作曲家加载音级序列
TRAIN_DIRS = [
    'Bach Johann Sebastian',
    'Ludwig van Beethoven',
    'Chopin Frederic',
    'Claude Debussy',
    'The Beatles',
    'ABBA',
    'Queen',
    'Duke Ellington',
    'Louis Armstrong',
    'Frank Sinatra',
]

all_sequences = []
for dirname in TRAIN_DIRS:
    seqs = load_pitch_sequences_from_dir(os.path.join(LMD_DIR, dirname))
    all_sequences.extend(seqs)
    print(f'{dirname}: {len(seqs)} 首')

total_notes = sum(len(s) for s in all_sequences)
print(f'\n共 {len(all_sequences)} 首，{total_notes} 个音符')

# 统计 12×12 转移频次矩阵
transition_counts_12 = np.zeros((12, 12), dtype=int)
for train_seq in all_sequences:
    for idx in range(len(train_seq) - 1):
        transition_counts_12[train_seq[idx], train_seq[idx+1]] += 1

# 归一化为转移概率
transition_prob_12 = transition_counts_12.astype(float)
row_sums = transition_prob_12.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
transition_prob_12 /= row_sums

print(f'\n12×12 转移矩阵统计完毕，总转移次数：{transition_counts_12.sum()}')

### 2.2 五声音阶转移矩阵热力图

从完整 12×12 转移频次矩阵中截取 C、D、E、G、A 对应的 5×5 子矩阵，再在该集合内按当前音逐行归一化。图中数值表示后继音仍属于该集合时的条件概率，而不是完整 12 音级转移概率矩阵的直接子矩阵。

> 这张图将用于书中正文（fig:markov-heatmap）。

**正文图题：在后继仍属于 C、D、E、G、A 的条件下重新归一化的一阶转移概率矩阵。**

In [ ]:
# 提取五声音阶子矩阵
penta_idx = PENTATONIC_PC  # [0, 2, 4, 7, 9]
transition_penta = transition_counts_12[np.ix_(penta_idx, penta_idx)].astype(float)

# 归一化
row_sums_p = transition_penta.sum(axis=1, keepdims=True)
row_sums_p[row_sums_p == 0] = 1
transition_penta_prob = transition_penta / row_sums_p

print('五声音阶 5×5 转移概率矩阵：')
print(f'{"":>8}', '  '.join(f'{n:>8}' for n in PENTATONIC_NAMES))
for row_i, penta_name in enumerate(PENTATONIC_NAMES):
    row = '  '.join(f'{transition_penta_prob[row_i,ci]:>8.3f}' for ci in range(5))
    print(f'{penta_name:>8}  {row}')

# 绘制 5×5 灰度热力图
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(transition_penta_prob, cmap='Greys', vmin=0, vmax=0.5,
               aspect='equal')

ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(PENTATONIC_NAMES, fontsize=11)
ax.set_yticklabels(PENTATONIC_NAMES, fontsize=11)
ax.set_xlabel('下一个音 $y_t$', fontsize=12)
ax.set_ylabel('当前音 $y_{t-1}$', fontsize=12)

# 在格子中标注数值
for ri in range(5):
    for ci in range(5):
        cell_val = transition_penta_prob[ri, ci]
        txt_color = 'white' if cell_val > 0.25 else 'black'
        ax.text(ci, ri, f'{cell_val:.3f}', ha='center', va='center',
                fontsize=11, color=txt_color, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label(r'$P(y_t \mid y_{t-1},\; y_t \in S)$', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_markov_heatmap.png'),
            dpi=600, bbox_inches='tight')
plt.show()
print('\nsaved → fig_markov_heatmap.png')

### 2.3 一阶前向生成

给定起始音，按转移概率逐步采样生成旋律。分别用两种五声音阶约束：
- **C D E G A**：本章按 C 宫五声音阶记名的无半音五音集合
- **C D F G A**：匹配本地《茉莉花》MIDI 实测音级集合的约束；不据此鉴定该文件的版本来源

两种约束都输出 MIDI 文件供聆听对比。

In [ ]:
def markov_generate_order1(trans_prob, start_pc, length,
                           allowed_pcs=None, seed=None):
    """一阶马尔可夫链前向生成。"""
    try:
        trans_prob = np.asarray(trans_prob, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError('trans_prob 必须是有限非负的 12×12 数值矩阵。') from exc
    if trans_prob.shape != (12, 12):
        raise ValueError('trans_prob 必须是 12×12 矩阵。')
    if not np.all(np.isfinite(trans_prob)) or np.any(trans_prob < 0):
        raise ValueError('trans_prob 必须只包含有限非负数。')
    if not _is_integer_scalar(start_pc) or not 0 <= int(start_pc) < 12:
        raise ValueError('start_pc 必须是 0--11 的整数。')
    if not _is_integer_scalar(length) or int(length) < 1:
        raise ValueError('length 必须是至少为 1 的整数。')
    start_pc = int(start_pc)
    length = int(length)
    if seed is not None:
        rng = np.random.RandomState(seed)
    else:
        rng = np.random.RandomState()

    if allowed_pcs is None:
        allowed_pcs = list(range(12))
    allowed_pcs = list(allowed_pcs)
    if (not allowed_pcs or
            any(not _is_integer_scalar(pc) or not 0 <= int(pc) < 12 for pc in allowed_pcs)):
        raise ValueError('allowed_pcs 必须是 0--11 中的非空整数音级集合。')
    allowed_pcs = list(dict.fromkeys(int(pc) for pc in allowed_pcs))

    if start_pc not in allowed_pcs:
        raise ValueError('start_pc 必须属于 allowed_pcs，避免起始音直接违反约束。')

    result = [start_pc]
    curr = start_pc
    for _ in range(length - 1):
        row_probs = trans_prob[curr].copy()
        prob_mask = np.zeros(12)
        for target_pc in allowed_pcs:
            prob_mask[target_pc] = row_probs[target_pc]
        prob_sum = prob_mask.sum()
        if not np.isfinite(prob_sum):
            raise ValueError('合法音级上的概率质量必须是有限数。')
        if prob_sum > 0:
            prob_mask /= prob_sum
        else:
            for target_pc in allowed_pcs:
                prob_mask[target_pc] = 1.0 / len(allowed_pcs)
        chosen = rng.choice(12, p=prob_mask)
        result.append(chosen)
        curr = chosen
    return result


# 小规模边界检查：确定性单位矩阵、非法状态/长度、负值与 NaN。
_identity_transition = np.eye(12, dtype=float)
assert markov_generate_order1(_identity_transition, 0, 3, seed=SEED) == [0, 0, 0]
_expect_value_error(markov_generate_order1, _identity_transition, 0.5, 3)
_expect_value_error(markov_generate_order1, _identity_transition, 0, 3.5)
_expect_value_error(markov_generate_order1, _identity_transition, 0, 3, allowed_pcs=[1, 2])
_negative_transition = _identity_transition.copy()
_negative_transition[0, 0] = -1.0
_expect_value_error(markov_generate_order1, _negative_transition, 0, 3)
_nan_transition = _identity_transition.copy()
_nan_transition[0, 0] = np.nan
_expect_value_error(markov_generate_order1, _nan_transition, 0, 3)

# 两种约束各生成 32 音
melody_cdega = markov_generate_order1(
    transition_prob_12, start_pc=0, length=32,
    allowed_pcs=PENTATONIC_PC, seed=SEED
)
melody_cdfga = markov_generate_order1(
    transition_prob_12, start_pc=0, length=32,
    allowed_pcs=JASMINE_PENTA_PC, seed=SEED
)

print('一阶生成（约束 C D E G A）：')
print(' '.join(pc_seq_to_names(melody_cdega)))
print()
print('一阶生成（约束 C D F G A）：')
print(' '.join(pc_seq_to_names(melody_cdfga)))

# 输出 MIDI
pitch_classes_to_midi(melody_cdega, out_path=os.path.join(MIDI_OUT_DIR, 'order1_cdega.mid'))
pitch_classes_to_midi(melody_cdfga, out_path=os.path.join(MIDI_OUT_DIR, 'order1_cdfga.mid'))

## 3. 高阶马尔可夫链

### 3.1 k 阶转移表构建

统计长度为 k 的上下文后接下一个音的频次，构建 k 阶转移表。

In [ ]:
def build_kth_order_table(input_seqs, order):
    """构建 k 阶转移表。

    返回 dict: tuple(上下文) -> Counter(下一个音 -> 频次)
    """
    if not _is_integer_scalar(order) or int(order) < 1:
        raise ValueError('order 必须是正整数。')
    order = int(order)
    trans_table = defaultdict(Counter)
    for s in input_seqs:
        for pos in range(order, len(s)):
            key = tuple(s[pos-order:pos])
            nxt = s[pos]
            trans_table[key][nxt] += 1
    return trans_table


def markov_generate_kth(trans_table, order, start_context, length,
                        allowed_pcs=None, seed=None):
    """k 阶马尔可夫链前向生成。

    参数：
        trans_table: k 阶转移表
        order: 阶数
        start_context: 起始上下文 tuple，长度为 order
        length: 生成序列总长度（含起始上下文）
        allowed_pcs: 约束音级列表（None 则不约束）
        seed: 随机种子
    """
    if not _is_integer_scalar(order) or int(order) < 1:
        raise ValueError('order 必须是正整数。')
    order = int(order)
    if len(start_context) != order:
        raise ValueError('start_context 长度必须等于 order。')
    if not _is_integer_scalar(length) or int(length) < order:
        raise ValueError('length 必须是且不能小于起始上下文长度的整数。')
    length = int(length)
    if seed is not None:
        rng = np.random.RandomState(seed)
    else:
        rng = np.random.RandomState()

    if allowed_pcs is None:
        allowed_pcs = list(range(12))
    allowed_pcs = list(allowed_pcs)
    if (not allowed_pcs or
            any(not _is_integer_scalar(pc) or not 0 <= int(pc) < 12 for pc in allowed_pcs)):
        raise ValueError('allowed_pcs 必须是 0--11 中的非空整数音级集合。')
    allowed_pcs = list(dict.fromkeys(int(pc) for pc in allowed_pcs))
    allowed_set = set(allowed_pcs)
    if any(not _is_integer_scalar(pc) or int(pc) not in allowed_set for pc in start_context):
        raise ValueError('start_context 中的每个音级都必须属于 allowed_pcs。')
    start_context = tuple(int(pc) for pc in start_context)

    result = list(start_context)
    for _ in range(length - order):
        key = tuple(result[-order:])
        next_counts = trans_table.get(key, Counter())
        if not next_counts:
            chosen = rng.choice(allowed_pcs)
            result.append(chosen)
            continue

        candidates = list(next_counts.keys())
        if any(not _is_integer_scalar(candidate) or not 0 <= int(candidate) < 12
               for candidate in candidates):
            raise ValueError('转移表的后继状态必须是 0--11 的整数音级。')
        weights = np.array([next_counts[candidate] for candidate in candidates], dtype=float)
        if not np.all(np.isfinite(weights)) or np.any(weights < 0):
            raise ValueError('转移表计数必须是有限非负数。')
        candidates = [int(candidate) for candidate in candidates]

        valid_mask = np.array([c in allowed_set for c in candidates])
        masked_weights = weights * valid_mask
        masked_sum = masked_weights.sum()
        if not np.isfinite(masked_sum):
            raise ValueError('合法后继的计数总和必须是有限数。')
        if masked_sum == 0:
            chosen = rng.choice(allowed_pcs)
        else:
            masked_weights /= masked_sum
            chosen = candidates[rng.choice(len(candidates), p=masked_weights)]

        result.append(chosen)
    return result


# 小规模边界检查：一阶上下文、非法长度与非有限计数。
_kth_test_table = {(0,): Counter({0: 1})}
assert markov_generate_kth(_kth_test_table, 1, (0,), 3, seed=SEED) == [0, 0, 0]
_expect_value_error(markov_generate_kth, _kth_test_table, 1, (0,), 3.5)
_bad_kth_test_table = {(0,): Counter({0: float('nan')})}
_expect_value_error(markov_generate_kth, _bad_kth_test_table, 1, (0,), 2)

# 构建 k=1, 2, 3 阶转移表
all_pc_sequences = list(all_sequences)

tables = {}
for cur_order in [1, 2, 3]:
    tables[cur_order] = build_kth_order_table(all_pc_sequences, cur_order)
    print(f'k={cur_order}: {len(tables[cur_order])} 种上下文')

print(f'\n转移表构建完毕。')

### 3.2 k=1, 2, 3 阶生成对比

用相同的起始音和长度，对比不同阶数生成的旋律（约束 C D E G A）。同时输出 MIDI 供聆听。

In [ ]:
# 使用相容的 C、C-D、C-D-E 起始上下文和相同总长度，对比 k=1, 2, 3 阶生成
GEN_LENGTH = 32
START_CONTEXTS = {
    1: (0,),       # 宫(C)
    2: (0, 2),     # 宫→商
    3: (0, 2, 4),  # 宫→商→角
}

generated_melodies = {}
for cur_order in [1, 2, 3]:
    gen_mel = markov_generate_kth(
        tables[cur_order], cur_order, START_CONTEXTS[cur_order], GEN_LENGTH,
        allowed_pcs=PENTATONIC_PC, seed=SEED
    )
    generated_melodies[cur_order] = gen_mel
    note_names = pc_seq_to_names(gen_mel)
    print(f'k={cur_order}: {" ".join(note_names)}')
    pitch_classes_to_midi(gen_mel,
                         out_path=os.path.join(MIDI_OUT_DIR, f'order{cur_order}_cdega.mid'))

# 绘制三条旋律的音高轮廓对比
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True, sharey=True)
for idx, cur_order in enumerate([1, 2, 3]):
    gen_mel = generated_melodies[cur_order]
    axes[idx].step(range(len(gen_mel)), gen_mel, where='mid',
                 color='black', linewidth=1.5)
    axes[idx].set_ylabel(f'k = {cur_order}', fontsize=12)
    axes[idx].set_yticks(PENTATONIC_PC)
    axes[idx].set_yticklabels(['C', 'D', 'E', 'G', 'A'], fontsize=10)
    axes[idx].grid(True, linestyle=':', alpha=0.4)
    axes[idx].set_xlim(-0.5, GEN_LENGTH - 0.5)

axes[2].set_xlabel('时间步', fontsize=12)
axes[0].set_title('不同阶数马尔可夫链生成的旋律（约束 C D E G A）', fontsize=13)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_markov_order_comparison.png'),
            dpi=600, bbox_inches='tight')
plt.show()
print('\nsaved → fig_markov_order_comparison.png')

## 4. 评估：直接在《茉莉花》上训练

前面的实验用本地艺术家目录子集训练转移矩阵，再用五声音阶约束来生成旋律。该目录不是官方 LMD 的 MD5 结构，项目中也没有来源映射或许可证，故不能把结果直接写成官方 Lakh MIDI Dataset 的统计。
这种表示的边界是：生成结果为等时值的 pitch-class 序列，不含节奏和八度信息。
而原曲在去除这些信息后也只剩音级序列；两者对比只能评价该 pitch-class 模型，不能代表包含音域与节奏的马尔可夫模型。

换一种方式：**直接在《茉莉花》上训练**，用 **（绝对音高，时值）** 联合 token 作为状态。
这样生成的序列保留节奏与绝对音高，可以输出为 MIDI，并与原曲比较音域、时值和局部转移。

### 4.1 从《茉莉花》提取 (pitch, duration) 序列

In [ ]:
jasmine_path = os.path.join(MELODY_DIR, '茉莉花.midi')
jasmine_pm = pretty_midi.PrettyMIDI(jasmine_path)
jasmine_notes = sorted(jasmine_pm.instruments[0].notes, key=lambda n: n.start)

# 提取 (pitch, duration) token 序列。duration 统一用四分音符时值数表示，通过 tempo map 把起止秒数分别换算为 tick。
jasmine_tokens = [
    (n.pitch, round((jasmine_pm.time_to_tick(n.end) - jasmine_pm.time_to_tick(n.start))
                    / jasmine_pm.resolution, 4))
    for n in jasmine_notes
]
tempo_times, jasmine_tempi = jasmine_pm.get_tempo_changes()
if len(jasmine_tempi) != 1:
    raise ValueError('输出示例假设全曲只有一个速度；检测到速度变化，请按 tempo map 写出。')
jasmine_tempo_bpm = float(jasmine_tempi[0])

print(f'《茉莉花》：{len(jasmine_tokens)} 个音符')
print(f'音域：{min(t[0] for t in jasmine_tokens)}–{max(t[0] for t in jasmine_tokens)} '
      f'({pretty_midi.note_number_to_name(min(t[0] for t in jasmine_tokens))}–'
      f'{pretty_midi.note_number_to_name(max(t[0] for t in jasmine_tokens))})')

# token 词表
unique_tokens = sorted(set(jasmine_tokens))
print(f'\n词表大小：{len(unique_tokens)} 种 (pitch, duration) token')
print(f'音高种类：{len(set(t[0] for t in jasmine_tokens))} 种')
print(f'时值种类：{sorted(set(t[1] for t in jasmine_tokens))}')

print(f'\n完整词表：')
for ut in unique_tokens:
    cnt = jasmine_tokens.count(ut)
    print(f'  ({pretty_midi.note_number_to_name(ut[0])}, {ut[1]:.2f}拍): {cnt} 次')

print(f'\n前 20 个 token：')
for idx, (pitch_val, dur_val) in enumerate(jasmine_tokens[:20]):
    print(f'  {idx:3d}: {pretty_midi.note_number_to_name(pitch_val):>4s}  dur={dur_val:.2f}拍')

### 4.2 在《茉莉花》上训练 k 阶 Markov chain

将 (pitch, duration) token 序列直接输入 `build_kth_order_table`。
由于只有 105 个 token，k=2 时上下文数量已接近序列长度，大量上下文只出现一次。

In [ ]:
# 构建 k=1, k=2 转移表
jasmine_tables = {}
for cur_order in [1, 2]:
    jasmine_tables[cur_order] = build_kth_order_table([jasmine_tokens], cur_order)
    ctx_table = jasmine_tables[cur_order]
    n_ctx = len(ctx_table)
    n_singletons = sum(1 for c in ctx_table.values() if sum(c.values()) == 1)
    print(f'k={cur_order}: {n_ctx} 种上下文，'
          f'其中 {n_singletons} 种只出现 1 次 ({n_singletons/n_ctx:.0%})')

print('\n--- k=1 转移表示例（前 5 个上下文）---')
for idx, (context, freq_counts) in enumerate(list(jasmine_tables[1].items())[:5]):
    pitch_val, dur_val = context[0]
    freq_total = sum(freq_counts.values())
    top3 = freq_counts.most_common(3)
    top3_str = ', '.join(
        f'{pretty_midi.note_number_to_name(tok_item[0])}({tok_item[1]:.2f}拍):{tok_cnt}/{freq_total}'
        for tok_item, tok_cnt in top3
    )
    print(f'  {pretty_midi.note_number_to_name(pitch_val)}({dur_val:.2f}拍) → {top3_str}')

### 4.3 生成与 MIDI 输出

用 k=1 和 k=2 各生成一条与原曲 token 数相同的旋律。生成器只查询最近 k 个 token；当前代码在未见上下文处回退到全词表抽样，且没有终止符，因此可以按调用者指定的长度继续生成。概率采样并不意味着每次运行必然不同：这里固定随机种子，所以相同环境与输入下输出可复现。

In [ ]:
def tokens_to_midi(token_seq, tempo_bpm=60.0, velocity=80, out_path=None):
    """将以四分音符时值数计时的 (pitch, duration) token 序列转为 MIDI。"""
    midi_obj = pretty_midi.PrettyMIDI(initial_tempo=tempo_bpm)
    inst = pretty_midi.Instrument(program=0)
    onset = 0.0
    for note_pitch, note_dur in token_seq:
        duration_seconds = note_dur * 60.0 / tempo_bpm
        inst.notes.append(pretty_midi.Note(
            velocity=velocity, pitch=note_pitch,
            start=onset, end=onset + duration_seconds))
        onset += duration_seconds
    midi_obj.instruments.append(inst)
    if out_path:
        midi_obj.write(out_path)
        print(f'  MIDI 已保存: {out_path}')
    return midi_obj


def generate_token_kth(trans_table, order, start_context, length,
                       fallback_tokens=None, seed=None):
    """k 阶 Markov chain 生成 (pitch, dur) token 序列。
    未见上下文回退时从 fallback_tokens 均匀采样；None 则取转移表中出现过的全部 token。
    """
    if not _is_integer_scalar(order) or int(order) < 1:
        raise ValueError('order 必须是正整数。')
    order = int(order)
    if len(start_context) != order:
        raise ValueError('start_context 长度必须等于 order。')
    if not _is_integer_scalar(length) or int(length) < order:
        raise ValueError('length 必须是且不能小于起始上下文长度的整数。')
    length = int(length)
    if fallback_tokens is None:
        fallback_tokens = sorted(
            {tok_item for context in trans_table for tok_item in context}
            | {tok_item for counts in trans_table.values() for tok_item in counts})
    else:
        fallback_tokens = list(fallback_tokens)
    if not fallback_tokens:
        raise ValueError('回退词表为空，无法执行未见上下文回退。')
    rng = np.random.RandomState(seed)
    result = list(start_context)
    for _ in range(length - order):
        key = tuple(result[-order:])
        next_counts = trans_table.get(key, Counter())
        if not next_counts:
            result.append(fallback_tokens[rng.randint(len(fallback_tokens))])
            continue
        candidates = list(next_counts.keys())
        weights = np.array([next_counts[tok_item] for tok_item in candidates], dtype=float)
        weight_sum = weights.sum()
        if (not np.all(np.isfinite(weights)) or np.any(weights < 0) or
                not np.isfinite(weight_sum) or weight_sum <= 0):
            raise ValueError('转移表计数必须是有限非负数且总和大于 0。')
        weights /= weight_sum
        result.append(candidates[rng.choice(len(candidates), p=weights)])
    return result


def fmt_token(note_tok):
    return f'{pretty_midi.note_number_to_name(note_tok[0]):>4s}({note_tok[1]:.2f})'


# 生成
gen_length = len(jasmine_tokens)

gen_k1 = generate_token_kth(
    jasmine_tables[1], 1,
    start_context=[jasmine_tokens[0]],
    length=gen_length, seed=SEED
)
gen_k2 = generate_token_kth(
    jasmine_tables[2], 2,
    start_context=jasmine_tokens[:2],
    length=gen_length, seed=SEED
)

# 输出 MIDI
tokens_to_midi(jasmine_tokens, tempo_bpm=jasmine_tempo_bpm,
               out_path=os.path.join(MIDI_OUT_DIR, 'jasmine_original.mid'))
tokens_to_midi(gen_k1, tempo_bpm=jasmine_tempo_bpm,
               out_path=os.path.join(MIDI_OUT_DIR, 'jasmine_gen_k1.mid'))
tokens_to_midi(gen_k2, tempo_bpm=jasmine_tempo_bpm,
               out_path=os.path.join(MIDI_OUT_DIR, 'jasmine_gen_k2.mid'))

# 打印前 20 个 token 对比
print('\n--- 前 20 个 token 对比 ---')
print(f'{"":>4s}  {"原曲":>12s}  {"k=1 生成":>12s}  {"k=2 生成":>12s}')
for idx in range(20):
    orig = fmt_token(jasmine_tokens[idx])
    g1 = fmt_token(gen_k1[idx])
    g2 = fmt_token(gen_k2[idx])
    print(f'{idx:3d}:  {orig}  {g1}  {g2}')

total_dur_orig = sum(d for _, d in jasmine_tokens)
total_dur_k1 = sum(d for _, d in gen_k1)
total_dur_k2 = sum(d for _, d in gen_k2)
print(f'\n总时长：原曲 {total_dur_orig:.1f}拍 / k=1 {total_dur_k1:.1f}拍 / k=2 {total_dur_k2:.1f}拍')
print(f'\n请聆听 output_midi/ 下的三个文件对比。')
print(f'当前固定 seed={SEED}；相同输入和环境下重新运行会复现同一结果。')

### 4.4 可视化对比

Piano-roll 对比：x 轴为时间（拍），y 轴为 MIDI 音高，矩形宽度反映时值。
统计对比：绝对音高分布、时值分布、音程分布。

In [ ]:
def plot_pianoroll(axis, token_seq, bar_color='black', alpha=0.8):
    """在 axis 上绘制 piano-roll。"""
    onset = 0.0
    for note_pitch, note_dur in token_seq:
        axis.barh(note_pitch, note_dur, left=onset, height=0.8,
                  color=bar_color, alpha=alpha,
                  edgecolor='white', linewidth=0.3)
        onset += note_dur
    return onset

# 图 1: Piano-roll 三行对比
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True, sharey=True)
titles = ['《茉莉花》原曲', 'k=1 生成', 'k=2 生成']
seqs = [jasmine_tokens, gen_k1, gen_k2]
colors = ['black', '#666666', '#999999']

for cur_ax, title_str, cur_seq, cur_color in zip(axes, titles, seqs, colors):
    end_time = plot_pianoroll(cur_ax, cur_seq, bar_color=cur_color)
    cur_ax.set_ylabel('MIDI 音高', fontsize=10)
    cur_ax.set_title(title_str, fontsize=11, loc='left')
    cur_ax.grid(True, axis='y', linestyle=':', alpha=0.3)
    seq_pitches = sorted(set(tok_item[0] for tok_item in cur_seq))
    cur_ax.set_yticks(seq_pitches)
    cur_ax.set_yticklabels([pretty_midi.note_number_to_name(p) for p in seq_pitches],
                       fontsize=8)

axes[2].set_xlabel('时间（拍）', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_markov_eval_pianoroll.png'),
            dpi=600, bbox_inches='tight')
plt.show()
print('saved → fig_markov_eval_pianoroll.png')

# 图 2: 统计对比（三列子图）
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 2a: 绝对音高分布
all_pitches_union = sorted(set(tok_item[0] for tok_item in jasmine_tokens) |
                           set(tok_item[0] for tok_item in gen_k1) |
                           set(tok_item[0] for tok_item in gen_k2))
pitch_labels = [pretty_midi.note_number_to_name(p) for p in all_pitches_union]
x_pos = np.arange(len(all_pitches_union))
width = 0.25

def pitch_hist(tokens):
    counter = Counter(tok_item[0] for tok_item in tokens)
    n_total = len(tokens)
    return [counter.get(p, 0) / n_total for p in all_pitches_union]

axes[0].bar(x_pos - width, pitch_hist(jasmine_tokens), width,
            color='black', alpha=0.8, label='原曲')
axes[0].bar(x_pos, pitch_hist(gen_k1), width,
            color='#888888', alpha=0.7, label='k=1')
axes[0].bar(x_pos + width, pitch_hist(gen_k2), width,
            color='#cccccc', alpha=0.9, label='k=2',
            edgecolor='#888888', linewidth=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(pitch_labels, fontsize=9, rotation=45)
axes[0].set_ylabel('归一化频率', fontsize=10)
axes[0].set_title('绝对音高分布', fontsize=11)
axes[0].legend(fontsize=8)
axes[0].grid(True, axis='y', linestyle=':', alpha=0.4)

# 2b: 时值分布
all_durs = sorted(set(tok_item[1] for tok_item in jasmine_tokens) |
                  set(tok_item[1] for tok_item in gen_k1) |
                  set(tok_item[1] for tok_item in gen_k2))
dur_labels = [f'{d:.2f}' for d in all_durs]
x_pos_d = np.arange(len(all_durs))

def dur_hist(tokens):
    counter = Counter(tok_item[1] for tok_item in tokens)
    n_total = len(tokens)
    return [counter.get(d, 0) / n_total for d in all_durs]

axes[1].bar(x_pos_d - width, dur_hist(jasmine_tokens), width,
            color='black', alpha=0.8, label='原曲')
axes[1].bar(x_pos_d, dur_hist(gen_k1), width,
            color='#888888', alpha=0.7, label='k=1')
axes[1].bar(x_pos_d + width, dur_hist(gen_k2), width,
            color='#cccccc', alpha=0.9, label='k=2',
            edgecolor='#888888', linewidth=0.8)
axes[1].set_xticks(x_pos_d)
axes[1].set_xticklabels(dur_labels, fontsize=9)
axes[1].set_xlabel('时值（拍）', fontsize=10)
axes[1].set_title('时值分布', fontsize=11)
axes[1].legend(fontsize=8)
axes[1].grid(True, axis='y', linestyle=':', alpha=0.4)

# 2c: 音程分布
def interval_hist(tokens, bins):
    ivls = [tokens[pos+1][0] - tokens[pos][0] for pos in range(len(tokens) - 1)]
    counter = Counter(ivls)
    n_total = len(ivls)
    if n_total == 0:
        return [0.0 for _ in bins]
    return [counter.get(b, 0) / n_total for b in bins]

all_interval_values = [
    cur_seq[pos + 1][0] - cur_seq[pos][0]
    for cur_seq in (jasmine_tokens, gen_k1, gen_k2)
    for pos in range(len(cur_seq) - 1)
]
i_vals = list(range(min(all_interval_values), max(all_interval_values) + 1))
axes[2].bar(np.array(i_vals) - width*0.5,
            interval_hist(jasmine_tokens, i_vals), width,
            color='black', alpha=0.8, label='原曲')
axes[2].bar(np.array(i_vals) + width*0.5,
            interval_hist(gen_k1, i_vals), width,
            color='#888888', alpha=0.7, label='k=1')
# k=2 用折线避免拥挤
k2_ihist = interval_hist(gen_k2, i_vals)
axes[2].plot(i_vals, k2_ihist, 'o-', color='#aaaaaa', markersize=3,
             linewidth=1.2, label='k=2')
axes[2].set_xlabel('音程（半音）', fontsize=10)
axes[2].set_title('音程分布', fontsize=11)
axes[2].legend(fontsize=8)
axes[2].grid(True, axis='y', linestyle=':', alpha=0.4)
axes[2].set_xlim(min(i_vals) - 1, max(i_vals) + 1)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_markov_eval_comparison.png'),
            dpi=600, bbox_inches='tight')
plt.show()
print('saved → fig_markov_eval_comparison.png')

# 统计摘要
print('\n--- 统计摘要 ---')
for label, cur_seq in [('原曲', jasmine_tokens), ('k=1', gen_k1), ('k=2', gen_k2)]:
    pitch_vals = [tok_item[0] for tok_item in cur_seq]
    dur_vals = [tok_item[1] for tok_item in cur_seq]
    ivl_vals = [cur_seq[pos+1][0] - cur_seq[pos][0] for pos in range(len(cur_seq)-1)]
    main_step_ratio = np.mean([abs(interval_value) in (2, 3) for interval_value in ivl_vals])
    print(f'{label}: {len(cur_seq)} 音, 总时长 {sum(dur_vals):.1f}拍, '
          f'音域 {min(pitch_vals)}-{max(pitch_vals)}, '
          f'平均音程 {np.mean(np.abs(ivl_vals)):.1f} 半音, '
          f'大二度/小三度占比 {main_step_ratio:.1%}')

print('注意：三条序列共享主要分布峰，但边际分布并不相同；由原曲训练也不自动保证边际分布一致。')
print('有限长度结果还受起始上下文、随机采样和未见上下文回退影响。')